# Dual-Teacher OWLv2 + GroundingDINO + SAM2 Verification Demo

Small validation notebook for dual-teacher pseudo-label verification before integrating anything into the main auto-labeling GUI.

Pipeline:

OWLv2 @ 0.12 + GroundingDINO @ 0.12 -> bbox matching/agreement -> merged bbox -> SAM2 segmentation -> CLIP verification -> high_quality / uncertain / rejected

## 2. Imports and configuration

In [ ]:
from __future__ import annotations

import json
import math
import random
import sys
import traceback
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    DEVICE = "cpu"


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "scripts" / "auto_label" / "auto_label_gui.py").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find repo root. Start Jupyter from prompt_video_segmenter or set REPO_ROOT manually.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

frames_root = Path(r"C:\Users\18447\Detector\data\kitchen_visor")
output_root = REPO_ROOT / "outputs" / "dual_teacher_demo"
debug_vis_root = output_root / "debug_vis"
output_root.mkdir(parents=True, exist_ok=True)
debug_vis_root.mkdir(parents=True, exist_ok=True)

OWLV2_THRESHOLD = 0.12
GROUNDINGDINO_THRESHOLD = 0.12
SAMPLE_SIZE = 20
RANDOM_SEED = 42
USE_VIDEO_RANDOM_FRAMES = False
VIDEO_ROOT = Path(r"C:\\Users\\18447\\Detector\\data\\egtea_gaze_plus\\video_clips\\cropped_clips")
VIDEO_SAMPLE_SIZE = 20
VIDEO_RANDOM_SEED = 123
VIDEO_FRAME_OUTPUT_DIR = output_root / "video_random_frames"
RUN_GROUNDINGDINO = True
RUN_VERIFIER = True
VERIFIER_MODEL_NAME = "openai/clip-vit-base-patch32"
VERIFIER_LOCAL_FILES_ONLY = False  # set True to avoid downloads

OWLV2_MODEL_DIR = REPO_ROOT / "models" / "owlv2-base-patch16"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Repo root : {REPO_ROOT}")
print(f"Frames    : {frames_root}")
print(f"Outputs   : {output_root}")
print(f"Device    : {DEVICE}")
print(f"OWLv2 threshold       : {OWLV2_THRESHOLD}")
print(f"GroundingDINO threshold: {GROUNDINGDINO_THRESHOLD}")
print(f"Verifier model         : {VERIFIER_MODEL_NAME}")

## 3. Class hierarchy

In [ ]:
KITCHEN_PROMPTS = [
    "hand", "glove",
    "pot", "pan", "lid", "cookware", "tray", "kettle",
    "bowl", "plate", "cup", "glass", "bottle", "jar",
    "container", "box", "package", "bag", "carton", "can",
    "knife", "fork", "spoon", "spatula", "tongs", "ladle", "whisk", "peeler", "scissors", "cutting board",
    "pasta", "noodles", "rice", "bread", "vegetable", "fruit", "meat", "fish", "egg", "cheese", "ingredient", "food", "dry food", "liquid", "water", "milk", "sauce", "oil", "powder", "sugar", "salt",
    "sink", "faucet", "stove", "cooktop", "oven", "microwave", "fridge", "drawer", "cabinet", "countertop", "table", "rack", "sponge", "towel",
]

INDUSTRIAL_PROMPTS = [
    "hand", "glove",
    "tool", "screwdriver", "wrench", "drill", "pliers",
    "screw", "bolt", "nut", "washer", "fastener",
    "panel", "cover", "plate", "bracket", "component", "part",
    "cable", "wire", "connector", "plug", "socket",
    "box", "tray", "bin", "package",
    "workbench", "table", "machine surface",
]

USE_PROMPT_SET = "kitchen"  # "kitchen" or "industrial"
TEXT_LABELS = KITCHEN_PROMPTS if USE_PROMPT_SET == "kitchen" else INDUSTRIAL_PROMPTS

FINE_TO_COARSE = {
    "hand": "hand", "glove": "hand",
    "pot": "cookware", "pan": "cookware", "lid": "cookware", "cookware": "cookware", "tray": "container", "kettle": "cookware",
    "bowl": "dishware", "plate": "dishware", "cup": "dishware", "glass": "dishware", "bottle": "container", "jar": "container",
    "container": "container", "box": "container", "package": "container", "bag": "container", "carton": "container", "can": "container",
    "knife": "utensil", "fork": "utensil", "spoon": "utensil", "spatula": "utensil", "tongs": "utensil", "ladle": "utensil", "whisk": "utensil", "peeler": "utensil", "scissors": "utensil", "cutting board": "utensil",
    "pasta": "ingredient", "noodles": "ingredient", "rice": "ingredient", "bread": "ingredient", "vegetable": "ingredient", "fruit": "ingredient", "meat": "ingredient", "fish": "ingredient", "egg": "ingredient", "cheese": "ingredient", "ingredient": "ingredient", "food": "ingredient", "dry food": "ingredient", "liquid": "ingredient", "water": "ingredient", "milk": "ingredient", "sauce": "ingredient", "oil": "ingredient", "powder": "ingredient", "sugar": "ingredient", "salt": "ingredient",
    "sink": "kitchen_scene", "faucet": "kitchen_scene", "stove": "kitchen_scene", "cooktop": "kitchen_scene", "oven": "kitchen_scene", "microwave": "kitchen_scene", "fridge": "kitchen_scene", "drawer": "kitchen_scene", "cabinet": "kitchen_scene", "countertop": "kitchen_scene", "table": "kitchen_scene", "rack": "kitchen_scene", "sponge": "kitchen_scene", "towel": "kitchen_scene",
    "tool": "tool", "screwdriver": "tool", "wrench": "tool", "drill": "tool", "pliers": "tool",
    "screw": "fastener", "bolt": "fastener", "nut": "fastener", "washer": "fastener", "fastener": "fastener",
    "panel": "part", "cover": "part", "bracket": "part", "component": "part", "part": "part",
    "cable": "electrical", "wire": "electrical", "connector": "electrical", "plug": "electrical", "socket": "electrical",
    "bin": "container", "workbench": "workspace", "machine surface": "workspace",
}


def normalize_label(label: str | None) -> str:
    return " ".join(str(label or "unknown").strip().lower().replace("-", " ").replace("_", " ").split())


def fine_to_coarse(label: str | None) -> str:
    label = normalize_label(label)
    return FINE_TO_COARSE.get(label, label if label else "unknown")


def display_label(label: str | None) -> str:
    fine = normalize_label(label)
    return f"{fine_to_coarse(fine)}:{fine}"


print(f"Prompt set: {USE_PROMPT_SET}, labels={len(TEXT_LABELS)}")
print(", ".join(TEXT_LABELS))

## 4. Load sample frames

In [ ]:
def list_images(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


all_frames = list_images(frames_root)
if not all_frames:
    raise FileNotFoundError(f"No frames found under {frames_root}. Expected extracted frames in data/auto_label_demo/frames.")

rng = random.Random(RANDOM_SEED)
sampled_frames = rng.sample(all_frames, min(SAMPLE_SIZE, len(all_frames)))
print(f"Found {len(all_frames)} frames; sampled {len(sampled_frames)}")
for idx, path in enumerate(sampled_frames[:5]):
    print(f"{idx:02d}: {path}")

### Optional input override: sample frames directly from local videos

Default is now `USE_VIDEO_RANDOM_FRAMES=False`, so Run All uses random images from `frames_root`, which points to local Kitchen VISOR screenshots. Enable this cell only if you explicitly want to replace them with decoded video screenshots.

In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}


def list_videos(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)


def sample_random_video_frames(video_root: Path, n_samples: int, seed: int, output_dir: Path) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    videos = list_videos(video_root)
    if not videos:
        raise FileNotFoundError(f"No videos found under {video_root}. Set VIDEO_ROOT to your local video directory, or set USE_VIDEO_RANDOM_FRAMES=False.")
    rng = random.Random(seed)
    sampled_output_paths: list[Path] = []
    attempts = 0
    max_attempts = max(50, n_samples * 8)

    while len(sampled_output_paths) < n_samples and attempts < max_attempts:
        attempts += 1
        video_path = rng.choice(videos)
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if frame_count <= 0:
            cap.release()
            continue
        frame_idx = rng.randrange(frame_count)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        cap.release()
        if not ok or frame is None:
            continue
        out_path = output_dir / f"video_sample_{len(sampled_output_paths):03d}_{video_path.stem}_f{frame_idx}.jpg"
        cv2.imwrite(str(out_path), frame)
        sampled_output_paths.append(out_path)

    if len(sampled_output_paths) < n_samples:
        print(f"[warn] Requested {n_samples} frames, extracted {len(sampled_output_paths)} after {attempts} attempts.")
    return sampled_output_paths


if USE_VIDEO_RANDOM_FRAMES:
    sampled_frames = sample_random_video_frames(
        VIDEO_ROOT,
        VIDEO_SAMPLE_SIZE,
        VIDEO_RANDOM_SEED,
        VIDEO_FRAME_OUTPUT_DIR,
    )
    print(f"Using {len(sampled_frames)} random video screenshots from {VIDEO_ROOT}")
    print(f"Saved extracted screenshots to {VIDEO_FRAME_OUTPUT_DIR}")
else:
    print(f"Using {len(sampled_frames)} pre-extracted frames from {frames_root}")

for idx, path in enumerate(sampled_frames[:5]):
    print(f"{idx:02d}: {path}")


## 5. OWLv2 detector wrapper

In [ ]:
_owlv2_state: dict[str, Any] | None = None
_owlv2_error = ""


def load_owlv2() -> dict[str, Any] | None:
    global _owlv2_state, _owlv2_error
    if _owlv2_state is not None:
        return _owlv2_state
    try:
        from transformers import Owlv2ForObjectDetection, Owlv2Processor

        model_path = str(OWLV2_MODEL_DIR if OWLV2_MODEL_DIR.exists() else "google/owlv2-base-patch16")
        processor = Owlv2Processor.from_pretrained(model_path)
        model = Owlv2ForObjectDetection.from_pretrained(model_path).to(DEVICE).eval()
        if DEVICE == "cuda":
            model = model.half()
        _owlv2_state = {"processor": processor, "model": model}
        print(f"OWLv2 loaded from {model_path}")
    except Exception as exc:
        _owlv2_error = f"{type(exc).__name__}: {exc}"
        print("OWLv2 unavailable. Install/prepare with: pip install transformers torch pillow")
        print(_owlv2_error)
        traceback.print_exc(limit=2)
        _owlv2_state = None
    return _owlv2_state


def run_owlv2(image: Image.Image, text_labels: list[str], threshold: float) -> list[dict[str, Any]]:
    state = load_owlv2()
    if state is None:
        return []
    labels = [normalize_label(x) for x in text_labels if normalize_label(x)]
    inputs = state["processor"](text=[labels], images=image, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    if DEVICE == "cuda" and "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].half()
    with torch.inference_mode():
        outputs = state["model"](**inputs)
    target_sizes = torch.tensor([(image.height, image.width)], device=DEVICE)
    processor = state["processor"]
    if hasattr(processor, "post_process_object_detection"):
        processed = processor.post_process_object_detection(outputs=outputs, threshold=threshold, target_sizes=target_sizes)[0]
    else:
        processed = processor.post_process_grounded_object_detection(outputs=outputs, threshold=threshold, target_sizes=target_sizes)[0]
    rows = []
    for box, score, label_idx in zip(processed["boxes"], processed["scores"], processed["labels"]):
        label_id = int(label_idx)
        rows.append({
            "bbox_xyxy": [float(v) for v in box.detach().cpu().tolist()],
            "label": labels[label_id] if 0 <= label_id < len(labels) else f"class_{label_id}",
            "score": float(score.detach().cpu()),
            "source": "owlv2",
        })
    return rows


load_owlv2()

## 6. GroundingDINO wrapper

In [ ]:
_dino_detector = None
_dino_error = ""


def load_groundingdino():
    global _dino_detector, _dino_error
    if not RUN_GROUNDINGDINO:
        _dino_error = "RUN_GROUNDINGDINO=False"
        return None
    if _dino_detector is not None:
        return _dino_detector
    try:
        from src.detection.grounding_dino import GroundingDINOPromptDetector

        cfg = {
            "detector": {
                "groundingdino_checkpoint_path": str(REPO_ROOT / "models" / "groundingdino_swint_ogc.pth"),
                "device": DEVICE,
                "confidence_threshold": GROUNDINGDINO_THRESHOLD,
                "box_threshold": GROUNDINGDINO_THRESHOLD,
                "text_threshold": GROUNDINGDINO_THRESHOLD,
                "groundingdino_per_label_enabled": False,
                "groundingdino_nms_iou_threshold": 0.45,
                "groundingdino_max_detections": 80,
                "groundingdino_max_per_label": 10,
                "groundingdino_cookware_merge_enabled": False,
            }
        }
        detector = GroundingDINOPromptDetector(cfg, REPO_ROOT, log=print)
        if not detector.available:
            raise RuntimeError(detector.warning or "GroundingDINO detector unavailable")
        _dino_detector = detector
        print(f"GroundingDINO loaded: {detector.backend_name}")
    except Exception as exc:
        _dino_error = f"{type(exc).__name__}: {exc}"
        print("GroundingDINO unavailable. Set RUN_GROUNDINGDINO=False to run OWLv2-only.")
        print(_dino_error)
        traceback.print_exc(limit=2)
        _dino_detector = None
    return _dino_detector


def run_groundingdino(image_bgr: np.ndarray, text_labels: list[str], threshold: float) -> list[dict[str, Any]]:
    detector = load_groundingdino()
    if detector is None:
        return []
    dets = detector.detect(image_bgr, frame_idx=0, prompt_labels=text_labels)
    rows = []
    for det in dets:
        score = float(getattr(det, "confidence", 0.0))
        if score < threshold:
            continue
        rows.append({
            "bbox_xyxy": [float(v) for v in det.bbox],
            "label": normalize_label(det.label),
            "score": score,
            "source": "groundingdino",
        })
    return rows


load_groundingdino()

## 7. BBox matching utilities

In [ ]:
def _area(box: list[float]) -> float:
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def bbox_iou(box_a: list[float], box_b: list[float]) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = _area([ix1, iy1, ix2, iy2])
    union = _area(box_a) + _area(box_b) - inter
    return inter / union if union > 0 else 0.0


def containment_ratio(box_a: list[float], box_b: list[float]) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter = _area([max(ax1, bx1), max(ay1, by1), min(ax2, bx2), min(ay2, by2)])
    smaller = min(_area(box_a), _area(box_b))
    return inter / smaller if smaller > 0 else 0.0


def center_distance_ratio(box_a: list[float], box_b: list[float]) -> float:
    acx, acy = (box_a[0] + box_a[2]) / 2, (box_a[1] + box_a[3]) / 2
    bcx, bcy = (box_b[0] + box_b[2]) / 2, (box_b[1] + box_b[3]) / 2
    diag = math.hypot(max(box_a[2] - box_a[0], box_b[2] - box_b[0]), max(box_a[3] - box_a[1], box_b[3] - box_b[1]))
    return math.hypot(acx - bcx, acy - bcy) / diag if diag > 0 else 1.0


def _is_match(a: dict, b: dict) -> tuple[bool, dict[str, float]]:
    iou = bbox_iou(a["bbox_xyxy"], b["bbox_xyxy"])
    contain = containment_ratio(a["bbox_xyxy"], b["bbox_xyxy"])
    center = center_distance_ratio(a["bbox_xyxy"], b["bbox_xyxy"])
    return (iou >= 0.35 or contain >= 0.60 or center <= 0.35), {"iou": iou, "containment": contain, "center_distance_ratio": center}


def match_detections(owl_dets: list[dict], dino_dets: list[dict]) -> tuple[list[dict], list[dict], list[dict]]:
    matches = []
    used_dino = set()
    for oi, owl in enumerate(owl_dets):
        best = None
        for di, dino in enumerate(dino_dets):
            if di in used_dino:
                continue
            ok, metrics = _is_match(owl, dino)
            if not ok:
                continue
            rank = (metrics["iou"], metrics["containment"], -metrics["center_distance_ratio"])
            if best is None or rank > best[0]:
                best = (rank, di, dino, metrics)
        if best is not None:
            _, di, dino, metrics = best
            used_dino.add(di)
            owl_label = normalize_label(owl["label"])
            dino_label = normalize_label(dino["label"])
            matches.append({
                "owl": owl,
                "dino": dino,
                "metrics": metrics,
                "fine_label_agreement": owl_label == dino_label,
                "coarse_label_agreement": fine_to_coarse(owl_label) == fine_to_coarse(dino_label),
            })
    unmatched_owl = [det for idx, det in enumerate(owl_dets) if all(m["owl"] is not det for m in matches)]
    unmatched_dino = [det for idx, det in enumerate(dino_dets) if idx not in used_dino]
    return matches, unmatched_owl, unmatched_dino

## 8. Candidate merge

In [ ]:
def union_with_padding(boxes: list[list[float]], image_w: int, image_h: int, padding_ratio: float = 0.08) -> list[float]:
    x1 = min(b[0] for b in boxes)
    y1 = min(b[1] for b in boxes)
    x2 = max(b[2] for b in boxes)
    y2 = max(b[3] for b in boxes)
    pad = padding_ratio * max(x2 - x1, y2 - y1)
    return [max(0.0, x1 - pad), max(0.0, y1 - pad), min(float(image_w - 1), x2 + pad), min(float(image_h - 1), y2 + pad)]


def confidence_tier(max_score: float) -> str:
    if max_score < 0.25:
        return "low"
    if max_score < 0.45:
        return "medium"
    return "high"


def make_candidate_from_dets(dets: list[dict], image_w: int, image_h: int, candidate_id: int, match_info: dict | None = None) -> dict:
    by_source = {d["source"]: d for d in dets}
    scores = [float(d["score"]) for d in dets]
    owl = by_source.get("owlv2")
    dino = by_source.get("groundingdino")
    preferred = owl or dino or dets[0]
    detector_agreement = len(by_source) >= 2
    coarse_agree = bool(match_info.get("coarse_label_agreement", False)) if match_info else False
    final_label = normalize_label(preferred["label"])
    return {
        "candidate_id": candidate_id,
        "bbox_xyxy": preferred["bbox_xyxy"],
        "merged_bbox_xyxy": union_with_padding([d["bbox_xyxy"] for d in dets], image_w, image_h),
        "source_detectors": sorted(by_source),
        "owlv2_label": normalize_label(owl["label"]) if owl else None,
        "owlv2_score": float(owl["score"]) if owl else None,
        "groundingdino_label": normalize_label(dino["label"]) if dino else None,
        "groundingdino_score": float(dino["score"]) if dino else None,
        "max_detector_score": max(scores),
        "mean_detector_score": float(np.mean(scores)),
        "detector_agreement": detector_agreement,
        "coarse_label_agreement": coarse_agree,
        "fine_label_agreement": bool(match_info.get("fine_label_agreement", False)) if match_info else False,
        "match_metrics": match_info.get("metrics", {}) if match_info else {},
        "confidence_tier": confidence_tier(max(scores)),
        "final_fine_label": final_label,
        "final_coarse_label": fine_to_coarse(final_label),
    }


def build_candidates(owl_dets: list[dict], dino_dets: list[dict], image_w: int, image_h: int) -> list[dict]:
    matches, unmatched_owl, unmatched_dino = match_detections(owl_dets, dino_dets)
    candidates = []
    cid = 0
    for match in matches:
        candidates.append(make_candidate_from_dets([match["owl"], match["dino"]], image_w, image_h, cid, match))
        cid += 1
    for det in unmatched_owl + unmatched_dino:
        candidates.append(make_candidate_from_dets([det], image_w, image_h, cid, None))
        cid += 1
    return candidates

## 9. SAM2 segmentation wrapper

In [ ]:
_sam2 = None
_sam2_available = False
_sam2_error = ""


def load_sam2():
    global _sam2, _sam2_available, _sam2_error
    if _sam2 is not None or _sam2_error:
        return _sam2
    try:
        from src.segmentation.sam2_segmenter import SAM2BoxSegmenter

        run_dir = output_root / "_sam2_tmp"
        run_dir.mkdir(parents=True, exist_ok=True)
        cfg = {
            "segmenter": {
                "sam2_checkpoint_path": str(REPO_ROOT / "models" / "sam2" / "sam2_hiera_tiny.pt"),
                "sam2_model_cfg": str(REPO_ROOT / "models" / "sam2" / "sam2_hiera_t.yaml"),
                "device": DEVICE,
                "min_mask_area": 50,
                "mask_min_detection_confidence": 0.0,
                "mask_refine_enabled": True,
                "mask_refine_close_kernel": 3,
                "mask_track_refresh_interval": 1,
            }
        }
        _sam2 = SAM2BoxSegmenter(cfg, run_dir, log=print)
        _sam2_available = _sam2.predictor is not None
        if not _sam2_available:
            _sam2_error = _sam2.warning or "SAM2 predictor is None"
    except Exception as exc:
        _sam2_error = f"{type(exc).__name__}: {exc}"
        _sam2 = None
        _sam2_available = False
        print(f"SAM2 unavailable, will use bbox fallback: {_sam2_error}")
    return _sam2


def bbox_fallback_mask(image_shape: tuple[int, int], bbox: list[float]) -> np.ndarray:
    h, w = image_shape
    mask = np.zeros((h, w), dtype=np.uint8)
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w - 1, x2), min(h - 1, y2)
    if x2 > x1 and y2 > y1:
        mask[y1:y2, x1:x2] = 1
    return mask


def run_sam2_or_fallback(image_bgr: np.ndarray, candidate_bbox: list[float], label: str = "object") -> tuple[np.ndarray, str]:
    sam2 = load_sam2()
    if sam2 is not None and _sam2_available:
        try:
            from src.common import Detection
            det = Detection(frame_idx=0, label=label, bbox=[float(v) for v in candidate_bbox], confidence=1.0, source="merged_candidate")
            masks = sam2.segment(image_bgr, [det], frame_idx=0, save_mask_pngs=False)
            if masks and masks[0].mask is not None:
                return masks[0].mask.astype(np.uint8), "sam2"
        except Exception as exc:
            print(f"SAM2 failed for candidate, using bbox fallback: {exc}")
    return bbox_fallback_mask(image_bgr.shape[:2], candidate_bbox), "bbox_fallback"


load_sam2()
print(f"SAM2 available={_sam2_available}, error={_sam2_error}")

## 10. Mask quality checks

In [ ]:
def mask_quality(mask: np.ndarray, bbox: list[float], image_w: int, image_h: int) -> dict[str, Any]:
    min_mask_area_ratio_image = 0.0002
    max_mask_area_ratio_image = 0.65
    min_mask_area_ratio_bbox = 0.15
    max_bbox_aspect_ratio = 8.0
    mask_area = float(mask.astype(bool).sum())
    image_area = float(image_w * image_h)
    bbox_area = max(1.0, _area(bbox))
    bw = max(1.0, bbox[2] - bbox[0])
    bh = max(1.0, bbox[3] - bbox[1])
    aspect = max(bw / bh, bh / bw)
    area_img = mask_area / image_area if image_area else 0.0
    area_box = mask_area / bbox_area if bbox_area else 0.0
    flags = []
    if area_img < min_mask_area_ratio_image:
        flags.append("mask_too_small_image")
    if area_img > max_mask_area_ratio_image:
        flags.append("mask_too_large_image")
    if area_box < min_mask_area_ratio_bbox:
        flags.append("mask_too_small_bbox")
    if aspect > max_bbox_aspect_ratio:
        flags.append("bbox_extreme_aspect")
    valid = not flags
    score = 1.0
    score -= 0.35 if "mask_too_small_image" in flags else 0.0
    score -= 0.35 if "mask_too_large_image" in flags else 0.0
    score -= 0.25 if "mask_too_small_bbox" in flags else 0.0
    score -= 0.20 if "bbox_extreme_aspect" in flags else 0.0
    return {
        "mask_area_ratio_image": area_img,
        "mask_area_ratio_bbox": area_box,
        "bbox_aspect_ratio": aspect,
        "mask_valid": valid,
        "mask_quality_score": max(0.0, score),
        "mask_quality_flags": flags,
    }

## 11. CLIP / SigLIP verifier

In [ ]:
_verifier = None
_verifier_error = ""


def load_verifier():
    global _verifier, _verifier_error
    if not RUN_VERIFIER:
        _verifier_error = "RUN_VERIFIER=False"
        return None
    if _verifier is not None or _verifier_error:
        return _verifier
    try:
        from transformers import CLIPModel, CLIPProcessor
        model = CLIPModel.from_pretrained(VERIFIER_MODEL_NAME, local_files_only=VERIFIER_LOCAL_FILES_ONLY).to(DEVICE).eval()
        processor = CLIPProcessor.from_pretrained(VERIFIER_MODEL_NAME, local_files_only=VERIFIER_LOCAL_FILES_ONLY)
        _verifier = {"model": model, "processor": processor, "labels": [normalize_label(x) for x in TEXT_LABELS]}
        print(f"Verifier loaded: {VERIFIER_MODEL_NAME}")
    except Exception as exc:
        _verifier_error = f"{type(exc).__name__}: {exc}"
        print("Verifier unavailable. Install/prepare: pip install transformers; model openai/clip-vit-base-patch32")
        print(_verifier_error)
        traceback.print_exc(limit=2)
        _verifier = None
    return _verifier


def masked_crop(image_rgb: np.ndarray, mask: np.ndarray, bbox: list[float]) -> Image.Image:
    h, w = image_rgb.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w - 1, x2), min(h - 1, y2)
    if x2 <= x1 or y2 <= y1:
        return Image.fromarray(image_rgb)
    crop = image_rgb[y1:y2, x1:x2].copy()
    crop_mask = mask[y1:y2, x1:x2].astype(bool)
    if crop_mask.shape[:2] == crop.shape[:2]:
        crop[~crop_mask] = 32
    return Image.fromarray(crop)


def verify_crop_zero_shot(crop: Image.Image, class_labels: list[str]) -> dict[str, Any]:
    verifier = load_verifier()
    labels = [normalize_label(x) for x in class_labels]
    if verifier is None:
        return {
            "verifier_top1_label": None,
            "verifier_top1_score": None,
            "verifier_top5_labels": [],
            "verifier_top5_scores": [],
            "verifier_margin": None,
            "verifier_coarse_label": None,
            "verifier_error": _verifier_error,
        }
    texts = [f"a photo of a {label}" for label in labels]
    inputs = verifier["processor"](text=texts, images=crop, return_tensors="pt", padding=True).to(DEVICE)
    with torch.inference_mode():
        outputs = verifier["model"](**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)[0].detach().cpu().numpy()
    order = np.argsort(-probs)[:5]
    top_labels = [labels[i] for i in order]
    top_scores = [float(probs[i]) for i in order]
    margin = top_scores[0] - (top_scores[1] if len(top_scores) > 1 else 0.0)
    return {
        "verifier_top1_label": top_labels[0] if top_labels else None,
        "verifier_top1_score": top_scores[0] if top_scores else None,
        "verifier_top5_labels": top_labels,
        "verifier_top5_scores": top_scores,
        "verifier_margin": float(margin),
        "verifier_coarse_label": fine_to_coarse(top_labels[0]) if top_labels else None,
        "verifier_error": "",
    }


load_verifier()

## 12. Verification decision logic

In [ ]:
def decide_quality(candidate: dict[str, Any]) -> dict[str, Any]:
    reasons = []
    votes = 0
    detector_agree = bool(candidate.get("detector_agreement"))
    coarse_agree = bool(candidate.get("coarse_label_agreement"))
    mask_pass = bool(candidate.get("mask_valid"))
    verifier_label = candidate.get("verifier_coarse_label")
    final_coarse = candidate.get("final_coarse_label")
    verifier_coarse_agree = bool(verifier_label and verifier_label == final_coarse)
    severe_coarse_conflict = bool(verifier_label and final_coarse and verifier_label != final_coarse and candidate.get("verifier_top1_score", 0) and candidate.get("verifier_top1_score", 0) > 0.35)

    for name, ok in [("detector_agreement", detector_agree), ("coarse_label_agreement", coarse_agree), ("verifier_coarse_agreement", verifier_coarse_agree), ("mask_quality_pass", mask_pass)]:
        if ok:
            votes += 1
            reasons.append(f"+{name}")
        else:
            reasons.append(f"-{name}")

    tier = candidate.get("confidence_tier", "low")
    if not mask_pass:
        status = "rejected"
        reasons.append("bad_mask")
    elif tier == "low":
        status = "high_quality" if detector_agree and coarse_agree and verifier_coarse_agree else "uncertain"
    elif tier == "medium":
        status = "high_quality" if votes >= 2 else "uncertain"
    else:
        if severe_coarse_conflict:
            status = "uncertain"
            reasons.append("severe_coarse_conflict")
        else:
            status = "high_quality"

    candidate["verification_votes"] = votes
    candidate["verification_status"] = status
    candidate["quality_status"] = status
    candidate["display_label"] = display_label(candidate.get("final_fine_label"))
    candidate["decision_reasons"] = reasons
    return candidate

## 13. Run demo and save debug visualizations

In [ ]:
def draw_debug(image_rgb: np.ndarray, owl_dets: list[dict], dino_dets: list[dict], candidates: list[dict], out_path: Path) -> None:
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(image_rgb)
    ax.axis("off")
    for det in owl_dets:
        x1, y1, x2, y2 = det["bbox_xyxy"]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="cyan", linewidth=1.0, linestyle="--"))
    for det in dino_dets:
        x1, y1, x2, y2 = det["bbox_xyxy"]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="orange", linewidth=1.0, linestyle=":"))
    for idx, cand in enumerate(candidates):
        mask = cand.get("mask")
        if mask is not None:
            overlay = np.zeros((*mask.shape, 4), dtype=float)
            color = plt.get_cmap("tab20")(idx % 20)[:3]
            overlay[mask.astype(bool), :3] = color
            overlay[mask.astype(bool), 3] = 0.28
            ax.imshow(overlay)
        x1, y1, x2, y2 = cand["merged_bbox_xyxy"]
        status = cand.get("quality_status", "?")
        edge = {"high_quality": "lime", "uncertain": "yellow", "rejected": "red"}.get(status, "white")
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=edge, linewidth=1.8))
        ax.text(x1, max(0, y1 - 4), f"{cand.get('display_label')} | {status}", color="black", fontsize=7, bbox={"facecolor": edge, "alpha": 0.85, "pad": 1})
    ax.set_title("cyan=OWLv2, orange=GroundingDINO, solid=merged candidate")
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)


all_records = []
for frame_idx, frame_path in enumerate(sampled_frames):
    print(f"[{frame_idx + 1:02d}/{len(sampled_frames)}] {frame_path}")
    image_bgr = cv2.imread(str(frame_path))
    if image_bgr is None:
        print("  could not read image")
        continue
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(image_rgb)
    h, w = image_bgr.shape[:2]

    owl_dets = run_owlv2(pil_image, TEXT_LABELS, OWLV2_THRESHOLD)
    dino_dets = run_groundingdino(image_bgr, TEXT_LABELS, GROUNDINGDINO_THRESHOLD)
    candidates = build_candidates(owl_dets, dino_dets, w, h)
    print(f"  owlv2={len(owl_dets)} dino={len(dino_dets)} candidates={len(candidates)}")

    for cand in candidates:
        mask, mask_source = run_sam2_or_fallback(image_bgr, cand["merged_bbox_xyxy"], cand.get("final_fine_label", "object"))
        cand["mask"] = mask
        cand["mask_source"] = mask_source
        cand.update(mask_quality(mask, cand["merged_bbox_xyxy"], w, h))
        crop = masked_crop(image_rgb, mask, cand["merged_bbox_xyxy"])
        cand.update(verify_crop_zero_shot(crop, TEXT_LABELS))
        decide_quality(cand)
        record = {k: v for k, v in cand.items() if k != "mask"}
        record["frame_path"] = str(frame_path)
        all_records.append(record)

    out_path = debug_vis_root / f"frame_{frame_idx:03d}.jpg"
    draw_debug(image_rgb, owl_dets, dino_dets, candidates, out_path)

print(f"Total candidates: {len(all_records)}")

## 14. JSONL export

In [ ]:
JSONL_FIELDS = [
    "frame_path", "candidate_id", "bbox_xyxy", "merged_bbox_xyxy", "source_detectors",
    "owlv2_label", "owlv2_score", "groundingdino_label", "groundingdino_score",
    "detector_agreement", "coarse_label_agreement", "confidence_tier", "mask_source",
    "mask_quality_score", "mask_quality_flags", "verifier_top1_label", "verifier_top1_score",
    "verifier_margin", "verification_votes", "quality_status", "final_fine_label",
    "final_coarse_label", "display_label", "decision_reasons",
]

jsonl_path = output_root / "dual_teacher_demo_results.jsonl"
with jsonl_path.open("w", encoding="utf-8") as fh:
    for rec in all_records:
        clean = {field: rec.get(field) for field in JSONL_FIELDS}
        fh.write(json.dumps(clean, ensure_ascii=False) + "\n")
print(f"Saved {len(all_records)} records to {jsonl_path}")

## 15. Summary table

In [ ]:
df = pd.DataFrame(all_records)
summary_rows = []

if len(df):
    for name, series in [
        ("quality_status", df["quality_status"].value_counts()),
        ("confidence_tier", df["confidence_tier"].value_counts()),
        ("final_coarse_label", df["final_coarse_label"].value_counts()),
    ]:
        for key, value in series.items():
            summary_rows.append({"metric": name, "key": key, "value": int(value)})
    summary_rows.append({"metric": "matched_dual_teacher_proposals", "key": "count", "value": int(df["detector_agreement"].sum())})
    summary_rows.append({"metric": "single_teacher_proposals", "key": "count", "value": int((~df["detector_agreement"].astype(bool)).sum())})
    summary_rows.append({"metric": "average_mask_quality_score", "key": "mean", "value": float(df["mask_quality_score"].mean())})

    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
    summary_csv = output_root / "summary.csv"
    summary_json = output_root / "summary.json"
    summary_df.to_csv(summary_csv, index=False)
    summary_json.write_text(json.dumps(summary_rows, indent=2), encoding="utf-8")
    print(f"Saved: {summary_csv}")
    print(f"Saved: {summary_json}")
else:
    summary_df = pd.DataFrame(columns=["metric", "key", "value"])
    display(summary_df)
    print("No records to summarize.")


## 16. Save final accepted bbox/mask views

After the demo run finishes, this cell creates two extra visualization sets for final candidates: bbox+mask overlays and mask-only views.

In [ ]:
FINAL_VIEW_STATUS_FILTER = {"high_quality"}  # use {"high_quality", "uncertain"} or None for all
final_bbox_mask_dir = output_root / "final_bbox_mask"
final_mask_only_dir = output_root / "final_mask_only"
final_bbox_mask_dir.mkdir(parents=True, exist_ok=True)
final_mask_only_dir.mkdir(parents=True, exist_ok=True)


def _candidate_visible(cand: dict[str, Any]) -> bool:
    if FINAL_VIEW_STATUS_FILTER is None:
        return True
    return cand.get("quality_status") in FINAL_VIEW_STATUS_FILTER


def _draw_final_bbox_mask(image_rgb: np.ndarray, candidates: list[dict[str, Any]], out_path: Path) -> None:
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(image_rgb)
    ax.axis("off")
    for idx, cand in enumerate(candidates):
        if not _candidate_visible(cand):
            continue
        mask = cand.get("mask")
        color = plt.get_cmap("tab20")(idx % 20)[:3]
        if mask is not None:
            overlay = np.zeros((*mask.shape, 4), dtype=float)
            overlay[mask.astype(bool), :3] = color
            overlay[mask.astype(bool), 3] = 0.38
            ax.imshow(overlay)
        x1, y1, x2, y2 = cand["merged_bbox_xyxy"]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.0))
        label = f"{cand.get('display_label')} | {cand.get('quality_status')}"
        ax.text(x1, max(0, y1 - 4), label, color="white", fontsize=8, bbox={"facecolor": color, "alpha": 0.85, "pad": 1})
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def _draw_final_mask_only(image_rgb: np.ndarray, candidates: list[dict[str, Any]], out_path: Path) -> None:
    canvas = np.zeros_like(image_rgb)
    any_mask = False
    for idx, cand in enumerate(candidates):
        if not _candidate_visible(cand):
            continue
        mask = cand.get("mask")
        if mask is None:
            continue
        any_mask = True
        color = np.asarray(plt.get_cmap("tab20")(idx % 20)[:3]) * 255
        canvas[mask.astype(bool)] = color.astype(np.uint8)
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(canvas if any_mask else np.zeros_like(image_rgb))
    ax.axis("off")
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# Rebuild per-frame candidate groups from runtime objects produced in section 13.
# This cell needs the previous demo-run cell to have completed, because JSONL does not store mask arrays.
final_view_rows = []
if "all_records" not in globals():
    raise RuntimeError("Run the main demo cell first so all_records/candidate masks exist in memory.")

# Re-run lightweight grouping from sampled frames and candidates by using the saved debug-time records if available.
# The main loop stores masks only in the local candidates list, so this cell repeats inference to recover masks for final views.
for frame_idx, frame_path in enumerate(sampled_frames):
    image_bgr = cv2.imread(str(frame_path))
    if image_bgr is None:
        continue
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(image_rgb)
    h, w = image_bgr.shape[:2]

    owl_dets = run_owlv2(pil_image, TEXT_LABELS, OWLV2_THRESHOLD)
    dino_dets = run_groundingdino(image_bgr, TEXT_LABELS, GROUNDINGDINO_THRESHOLD)
    candidates = build_candidates(owl_dets, dino_dets, w, h)
    for cand in candidates:
        mask, mask_source = run_sam2_or_fallback(image_bgr, cand["merged_bbox_xyxy"], cand.get("final_fine_label", "object"))
        cand["mask"] = mask
        cand["mask_source"] = mask_source
        cand.update(mask_quality(mask, cand["merged_bbox_xyxy"], w, h))
        crop = masked_crop(image_rgb, mask, cand["merged_bbox_xyxy"])
        cand.update(verify_crop_zero_shot(crop, TEXT_LABELS))
        decide_quality(cand)

    visible_count = sum(1 for c in candidates if _candidate_visible(c))
    out_bbox = final_bbox_mask_dir / f"frame_{frame_idx:03d}_bbox_mask.jpg"
    out_mask = final_mask_only_dir / f"frame_{frame_idx:03d}_mask_only.jpg"
    _draw_final_bbox_mask(image_rgb, candidates, out_bbox)
    _draw_final_mask_only(image_rgb, candidates, out_mask)
    final_view_rows.append({
        "frame_path": str(frame_path),
        "visible_candidates": visible_count,
        "bbox_mask_path": str(out_bbox),
        "mask_only_path": str(out_mask),
    })

final_view_df = pd.DataFrame(final_view_rows)
final_view_csv = output_root / "final_view_summary.csv"
final_view_df.to_csv(final_view_csv, index=False)
display(final_view_df)
print(f"Saved bbox+mask views to: {final_bbox_mask_dir}")
print(f"Saved mask-only views to: {final_mask_only_dir}")
print(f"Saved summary: {final_view_csv}")

## 16. How to interpret results

- Low-threshold detections are candidate proposals, not direct training labels.
- Dual-teacher agreement is used to increase precision before accepting labels.
- SAM2 or bbox fallback mask quality controls whether a proposal can be used for training.
- Verifier disagreement should send samples to `uncertain` instead of directly into training.
- Inspect `outputs/dual_teacher_demo/debug_vis/` before trusting the JSONL. This notebook is a validation demo only.